#Some explication
-PostgreSQL = SGBD (Service main)
-postgres = user so with role 'postgres' (superuser)
    -psql > client cli (PostgreSQL server client)

#Some commands
\du - list of roles
\dt - show tables
\d - tables, views and sequences
\l - show databases
\c database - connect database
\d table_name - table description (desc)
\conninfo - connection details




# Project Idea
1) Ingest data from yfinance
2) Extract and transform the data
3) Validate data using Pydantic and Pandas
4) Load data into PostgreSQL

In [ ]:

#imports
import yfinance
import pandas as pd
import json
from sqlalchemy import create_engine, Column, Integer, VARCHAR, DATETIME
from sqlalchemy.orm import declarative_base
from pydantic import SecretStr, Field
from pydantic_settings import BaseSettings, SettingsConfigDict

In [ ]:
dt_stocks = yfinance.download("NVDA AMZN", period="1mo", interval="1d")

#print(dt_stocks.columns)  # show column names
#print(dt_stocks.dtypes)   # show column data types
#print(dt_stocks.size)     # show total number of elements (rows × columns, including null values)
#print(dt_stocks.count())  # show the number of non-null values per column
#print(dt_stocks)

# Using .xs - method specifically for MultiIndex
#.xs() selects a part of the DataFrame quickly, without complex filters
#.xs() / MultiIndex example:
#Column level 0 -> Price (type of data)
#Column level 1 -> Ticket (e.g., NVDA)
#Data rows -> actual values

#axis=0 -> rows
#axis=1 -> columns
#level=0 -> Price
#level=1 -> Ticket

#important
#index -> column = amzn_stocks.reset_index()
#.filter - just to find column name or index

In [ ]:
#NVDA STOCKS
nvda_stocks = dt_stocks.xs('NVDA', axis=1, level=1)

#without TICKET level and date index

nvda_stocks = nvda_stocks.reset_index()

filter_nvda = nvda_stocks.Date.max()
max_date_nvda_stocks = nvda_stocks[nvda_stocks.Date==filter_nvda]

#date format -> max_date_nvda_stocks.iloc[0].Date.strftime("%d/%m/%Y")

In [ ]:
#AMZN STOCKS
amzn_stocks = dt_stocks.xs('AMZN', axis=1, level=1)

amzn_stocks = amzn_stocks.reset_index()

#filter
amzn_filter = amzn_stocks.Date.max()
max_date_amzn_stocks = amzn_stocks[amzn_stocks.Date==amzn_filter]

#date format -> max_date_amzn_stocks.iloc[0].Date.strftime("%d/%m/%Y")


In [ ]:
# Environment variables
class Environment(BaseSettings):
    
    model_config = SettingsConfigDict(env_file=".env_0001_postgresql_alchemy")
    
    db_user: SecretStr = Field(..., description="Username for connecting to the database", alias="DATABASE_USER")
    db_password: SecretStr = Field(..., description="Password for the database user", alias="DATABASE_PASSWORD")
    db_name: SecretStr = Field(..., description="Name of the database to connect to", alias="DATABASE_NAME")
    db_address: SecretStr = Field(..., description="Host or IP address of the database",alias="DATABASE_ADDRESS")
    
env = Environment()